# Make More Interesting Penguin Dataset

In [1]:
import numpy as np
import pandas as pd
import json

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
df.head(3)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE


## Augment Penguin Example

### Create an id

In [3]:
from utils import make_random_unique_id
# Maximum 3 contribution per penguin.
# A same penguin "species", "island", "sex" are the same through their life.
df = make_random_unique_id(
    df,
    id_column = "penguin_id",
    fixed_fields = ["species", "island", "sex"],
    max_contributions = 3
)

/home/onyxia/work/csvw-safe/csvw-safe-library/examples/utils.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(fixed_fields, dropna=False, group_keys=False).apply(random_merge)


### Sex as boolean

In [4]:
df["sex"] = df["sex"].map({"MALE": 1, "FEMALE": 0}).astype(bool)

### Add a timestamp (in days)

In [5]:
np.random.seed(42)
start = pd.Timestamp("2025-01-01")
end = pd.Timestamp("2025-12-31")

df["timestamp"] = start + pd.to_timedelta(
    np.random.randint(0, (end - start).days, size=len(df)),
    unit="D"
)

### Add another type of timestamp (in seconds)
After first timestamp

In [6]:
df["timestamp_with_time"] = df["timestamp"] + pd.to_timedelta(
    np.random.randint(0, 24*60*60, size=len(df)),  # seconds in a day
    unit="s"
)

### Add a favourite number between 0 and 5 (categorical int)

In [7]:
df["favourite_number"] = np.random.randint(0, 6, size=len(df))

### Put body_mass_g as continuous integer

In [8]:
df["body_mass_g"] = df["body_mass_g"].astype("Int64")

### Add nulls in bill_length_mm

In [9]:
# Randomly pick 100 indices to set as NaN
nan_indices = np.random.choice(df.index, size=100, replace=False)
df.loc[nan_indices, "bill_length_mm"] = np.nan
print(df["bill_length_mm"].isna().sum())  # Should print > 100

101


### Add nulls in flipper_length_mm (if in bill_length_mm and more)

In [10]:
# Add nulls in flipper_length_mm where bill_length_mm is null
df.loc[df["bill_length_mm"].isna(), "flipper_length_mm"] = np.nan

# Add 50 more random nulls in flipper_length_mm (excluding already nulls)
available_indices = df.index[df["flipper_length_mm"].notna()]
nan_indices_flipper_extra = np.random.choice(available_indices, size=50, replace=False)
df.loc[nan_indices_flipper_extra, "flipper_length_mm"] = np.nan
print(df[["bill_length_mm", "flipper_length_mm"]].isna().sum())

bill_length_mm       101
flipper_length_mm    151
dtype: int64


### Result

In [11]:
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750,True,5,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800,False,12,2025-12-15,2025-12-15 18:15:26,2


In [12]:
df.to_csv("penguin_plus.csv", header=True, index=False)

In [13]:
series = df["island"]
non_null = series.dropna()
r = sorted(non_null.unique())
r

['Biscoe', 'Dream', 'Torgersen']

In [69]:
col = "penguin_id"
column_name = "sex"
valid = df[[column_name, col]].dropna()
max_mapping_keys = 25
max_mapping_values = 10

In [70]:
valid.groupby(col)[column_name].apply(lambda x: list(pd.unique(x)))

penguin_id
2      [False]
5       [True]
8       [True]
10      [True]
12     [False]
        ...   
331     [True]
333     [True]
336     [True]
337     [True]
339     [True]
Name: sex, Length: 121, dtype: object